# POC Extraction de données

Extraction des données provenant des rapport de G2K vers un format plus simple.

Import, définition des variables et récupération du contenue du rapport (sous forme de `string`)

In [ ]:
import pandas as pd
import numpy as np
import re

## Variable global
content = ""
sections_content = {}
sections_titles = []

# Récuperaction du contenue du rapport
with open("../data/rapport-RGU_3cm.txt", "r") as f:
    content = f.read()

Définition de mes *regex*

In [ ]:
section_header_pattern          = re.compile(r"^\*+$\n^\*{5}(.*)\*{5}$\n^\*+$", re.MULTILINE)
s1_kv_pattern                   = re.compile(r"^([A-ZÀ-Ÿ][a-zA-ZÀ-ÿ' ]+?)\s*:\s*(.*)$", re.MULTILINE)
s2_header_pattern               = re.compile(r"(Numéro)\s+(Début)\s+-\s+(Fin)\s+(Centroïde)\s+(Energie)\s+(FWHM)\s+(Surface)\s+(Incert\.)\s+(Fond sous)\s*\r?\n^\s*(du pic)\s+(\(canaux\))\s+(\(keV\))\s+(\(keV\))\s+(le pic)", re.MULTILINE)
s2_data_pattern                 = re.compile(r"^\s*([MmF]?)\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+-\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)", re.MULTILINE)
s3_header_pattern               = re.compile(r"^\s+(Nom\sdu)\s+(Indice\sde)\s+(Energie)\s+(Intensité)\s+(Activité)\s+(Incert\.)$\n^\s+(nucléide)\s+(confiance)\s+(\W\w+\W)\s+(\W%\W)\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)\n", re.MULTILINE)
s3_data_pattern                 = re.compile(r"^\s*([A-Z]{1,2}-\d{1,3})?\s*(\d+\.\d+)?\s*(\d+\.\d+)(\*?)\s*(@?)\s*(\d+\.\d+)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)$", re.MULTILINE)
s4_header_1_pattern             = re.compile(r"^\s*(Nom du)\s+(Indice de)\s+(Activité moyenne)\s+(Incert\.)$\n^\s+(nucléide)\s+(confiance)\s+(pondérée)$\n^\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)$", re.MULTILINE)
s4_header_2_pattern             = re.compile(r"^\s*(Numéro)\s+(Energie)\s+(Intensité)\s+(Incert\.)\s+(Type)\s+(Nucléide)$\n^\s+(du pic)\s+(\WkeV\W)\s+(\Wcoups\Wsec\W)\s+(du pic)\s+(potentiel)$", re.MULTILINE)
s4_data_1_pattern               = re.compile(r"^\s*(X?)\s+([A-Z]{1,2}-\d{1,3})\s*(@?)\s+(\d+\.\d+)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)?\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)?", re.MULTILINE)
s4_data_2_pattern               = re.compile(r"^\s+([MmF])?\s*(\d+)\s+(\d+\.\d+)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+\.\d+)\s*(Sum|D-Esc\.|S-Esc\.|Tol\.)?\s*([A-Z]{1,2}-\d{1,3})?\s*$", re.MULTILINE)
s5_header_pattern               = re.compile(r"^\s+(Nom\sdu)\s+(Energie)\s+(Intensité)\s*(LD\sEnergie)\s*(LD\snucléide)\s+(Activité)\s+(SD\sEnergie)$\n^\s+(nucléide)\s+(\WkeV\W)\s+(\W%\W)\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)\s+(\WmBq\Wg\s+\W)\s+\s+(\WmBq\Wg\s+\W)$", re.MULTILINE)
s5_data_pattern                 = re.compile(r"^\s*(\+?)\s*(\??)\s*(>?)\s*([A-Z]{1,2}-\d{1,3})?\s+(\d+\.\d+)(\*?)\s*(@?)\s+(\d+\.\d+)\s+([+\-]?\d+(?:\.\d+)?(?:E[+\-]?\d+)?|Non\sCalc)(?:\s*([+\-]?\d+(?:\.\d+)?(?:E[+\-]?\d+)?))?\s+([+\-]?\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+([+\-]?\d+(?:\.\d+)?(?:E[+\-]?\d+)?)$", re.MULTILINE)
s6_header_pattern               = re.compile(r"^\s+(.*)$\n^\s+(.*)$", re.MULTILINE)
s6_word_column_pattern          = re.compile(r"([A-Za-zÀ-ÿ]+\.?)")
s6_nucleide_line_pattern        = re.compile(r"^[+>?]\s+([A-Z]{1,2}-\d{1,3})\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)", re.MULTILINE)


## Extraction des données de chaque section

### Notes
**TODO**
- Pour les sections de 2 à 6 extraire les métadonnées.
- Faire un `ffill` pour les noms de nucléide

### Setup
1 - On extrait les titre et on créer un dictionnaire de DataFrame

2- On découpe `content` par section que l'on range dans un dictionnaire `sections_content`

In [ ]:
section_headers = section_header_pattern.findall(content)

# 1
for i, section in enumerate(section_headers):
    sections_titles.append(section.strip())

# 2
matches = list(section_header_pattern.finditer(content))
for i, match in enumerate(matches):
    title = match.group(1).strip()
    start = match.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(content)
    sections_content[title] = content[start:end].strip()

### S1 - RAPPORT DE L’ANALYSE DU SPECTRE
C'est les metadata du rapport.

Pour l'instant ce n'est pas encore bon:
Mauvaise dections des clés/valeurs

In [ ]:
content_s1 = sections_content[sections_titles[0]]
matches = s1_kv_pattern.findall(content_s1)
data_s1 = {key.strip(): value.strip() for key, value in matches}

####### DATA ##########
data_s1

### S2 - RAPPORT ANALYSE DES PICS

In [ ]:
def extract_header_s2(content):
    match = re.search(s2_header_pattern, content)
  
    if not match:
        return None
        
    columns = [
        "Marker",                                   # Marqueur (M, m, F)
        f"{match.group(1)} {match.group(9)}",       # Numéro du pic
        f"{match.group(2)} {match.group(10)}",        # Début (canaux)       
        f"{match.group(3)} {match.group(11)}",                       # Fin (canaux)
        match.group(4),                             # Centroïde
        f"{match.group(5)} {match.group(12)}",      # Energie (keV)
        f"{match.group(6)} {match.group(13)}",      # FWHM (keV)
        match.group(7),                             # Surface
        match.group(8),                             # Incert.
        f"{match.group(8)} {match.group(13)}"       # Fond sous le pic
    ]
    
    return columns



In [ ]:
def extract_data_s2(content, header):
    matches = re.findall(s2_data_pattern, content)
    data = pd.DataFrame(matches, columns=header)
    data[header[1:]] = data[header[1:]].apply(pd.to_numeric, errors="coerce")
    
    return data


In [ ]:
content_s2 = sections_content[sections_titles[1]]
header_s2 = extract_header_s2(content_s2)
data_s2 = extract_data_s2(content_s2, header_s2)

####### DATA ##########
data_s2

### S3 - RAPPORT IDENTIFICATION DES NUCLEIDES

In [ ]:
def extract_header_s3(content):
    matches = re.search(s3_header_pattern, content)

    if not matches:
        return None
    
    columns = [
        f"{matches.group(1)} {matches.group(7)}",
        f"{matches.group(2)} {matches.group(8)}",
        f"{matches.group(3)} {matches.group(9)}",
        "Marker *",
        "Marker @",
        f"{matches.group(4)} {matches.group(10)}",
        f"{matches.group(5)} {matches.group(11)}",
        f"{matches.group(6)} {matches.group(12)}"
    ]
   
    return columns

In [ ]:
def extract_data_s3(content, header):
    matches = re.findall(s3_data_pattern, content)
    df = pd.DataFrame(matches ,columns=header)
    
    # Loaded variable 'data_s3' from kernel state

    # Replace all instances of empty string with Nan in columns: 'Nom du nucléide', 'Indice de confiance'
    df['Nom du nucléide'] = df['Nom du nucléide'].replace('', np.nan)
    df['Indice de confiance'] = df['Indice de confiance'].replace('', np.nan)

    # Change column type to float64 for columns: 'Energie (keV)', 'Intensité (%)' and 3 other columns
    df = df.astype({'Energie (keV)': 'float64', 'Intensité (%)': 'float64', 'Activité (mBq/g   )': 'float64', 'Incert. (mBq/g   )': 'float64', 'Indice de confiance': 'float64'})

    # Replace gaps forward from the previous valid value in: 'Nom du nucléide'
    df = df.fillna({'Nom du nucléide': df['Nom du nucléide'].ffill()})

    # Change column type to category for column: 'Nom du nucléide'
    df = df.astype({'Nom du nucléide': 'category'})

    # Change column type to bool for columns: 'Marker *', 'Marker @'
    df = df.astype({'Marker *': 'bool', 'Marker @': 'bool'})
    
    return df

In [ ]:
content_s3 = sections_content[sections_titles[2]]
header_s3 = extract_header_s3(content_s3)
data_s3 = extract_data_s3(content_s3, header_s3)

data_s3

### S4 - RAPPORT IDENTIFICATION AVEC CORRECTION D’INTERFERENCE

In [ ]:
def extract_header_1_s4(content):
    matches = re.search(s4_header_1_pattern, content)
    if not matches:
        return None
    
    columns = [
        "Marker (X)",
        f"{matches.group(1)} {matches.group(5)}",
        "Marker (@)",
        f"{matches.group(2)} {matches.group(6)}",
        f"{matches.group(3)} {matches.group(7)} {matches.group(8)}",
        f"{matches.group(4)} {matches.group(9)}"
    ]
    
    return columns

In [ ]:
def extract_header_2_s4(content):
    matches = re.search(s4_header_2_pattern, content)

    if not matches:
        return None

    columns = [
        "Marker (M/m/F)",
        f"{matches.group(1)} {matches.group(7)}",
        f"{matches.group(2)} {matches.group(8)}",
        f"{matches.group(3)} {matches.group(9)}",
        f"{matches.group(4)}",
        f"{matches.group(5)} {matches.group(10)}",
        f"{matches.group(6)} {matches.group(11)}",
    ]

    return columns

In [ ]:
def extract_data_1_s4(content, header):
    matches = re.findall(s4_data_1_pattern, content)
    df = pd.DataFrame(matches, columns=header)
    
    # Change column type to bool for columns: 'Marker (X)', 'Marker (@)'
    df = df.astype({'Marker (X)': 'bool', 'Marker (@)': 'bool'})

    # Replace all instances of empty string with NaN in columns: 'Activité moyenne pondérée (mBq/g   )', 'Incert. (mBq/g   )'
    df['Activité moyenne pondérée (mBq/g   )'] = df['Activité moyenne pondérée (mBq/g   )'].replace('', np.nan)
    df['Incert. (mBq/g   )'] = df['Incert. (mBq/g   )'].replace('', np.nan)

    # Change column type to object for columns: 'Indice de confiance', 'Activité moyenne pondérée (mBq/g   )', 'Incert. (mBq/g   )'
    df = df.astype({'Indice de confiance': 'object', 'Activité moyenne pondérée (mBq/g   )': 'object', 'Incert. (mBq/g   )': 'object'})

    # Change column type to category for column: 'Nom du nucléide'
    df = df.astype({'Nom du nucléide': 'category'})
    
    return df

In [ ]:
def extract_data_2_s4(content, header):
    matches = re.findall(s4_data_2_pattern, content)
    df = pd.DataFrame(matches, columns=header)
    

    # Replace all instances of empty string with NaN in columns: 'Marker (M/m/F)', 'Type du pic', 'Nucléide potentiel'
    df['Marker (M/m/F)'] = df['Marker (M/m/F)'].replace('', np.nan)
    df['Type du pic'] = df['Type du pic'].replace('', np.nan)
    df['Nucléide potentiel'] = df['Nucléide potentiel'].replace('', np.nan)

    # Change column type to category for columns: 'Marker (M/m/F)', 'Type du pic', 'Nucléide potentiel'
    df = df.astype({'Marker (M/m/F)': 'category', 'Type du pic': 'category', 'Nucléide potentiel': 'category'})

    # Change column type to float64 for columns: 'Numéro du pic', 'Energie (keV)' and 2 other columns
    df = df.astype({'Numéro du pic': 'float64', 'Energie (keV)': 'float64', 'Intensité (coups/sec)': 'float64', 'Incert.': 'float64'})
    
    return df

In [ ]:
content_s4 = sections_content[sections_titles[3]]
header_1_s4 = extract_header_1_s4(content_s4)
header_2_s4 = extract_header_2_s4(content_s4)

data_1_s4 = extract_data_1_s4(content_s4, header_1_s4)
data_2_s4 = extract_data_2_s4(content_s4, header_2_s4)

data_2_s4

### S5 - RAPPORT LIMITES DE DETECTION

In [ ]:
def extract_header_s5(content):
    matches = re.search(s5_header_pattern, content)
    
    if not matches:
        return None
    
    columns = [
        "Marker (+)",
        "Marker (?)",
        "Marker (>)",
        f"{matches.group(1)} {matches.group(8)}",
        f"{matches.group(2)} {matches.group(9)}",
        "Marker (*)",
        "Marker (@)",
        f"{matches.group(3)} {matches.group(10)}",
        f"{matches.group(4)} {matches.group(11)}",
        f"{matches.group(5)} {matches.group(12)}",
        f"{matches.group(6)} {matches.group(13)}",
        f"{matches.group(7)} {matches.group(14)}"
    ]
    
    return columns

In [ ]:
def extract_data_s5(content, header):
    matches = re.findall(s5_data_pattern, content)
    df = pd.DataFrame(matches, columns=header)
    
    # Loaded variable 'data_s5' from kernel state

    # Replace all instances of empty string with NaN in column: 'Nom du nucléide'
    df['Nom du nucléide'] = df['Nom du nucléide'].replace('', np.nan)

    # Replace all instances of empty string with NaN in column: 'LD nucléide (mBq/g   )'
    df['LD nucléide (mBq/g   )'] = df['LD nucléide (mBq/g   )'].replace('', np.nan)

    # Change column type to bool for columns: 'Marker (+)', 'Marker (?)' and 3 other columns
    df = df.astype({'Marker (+)': 'bool', 'Marker (?)': 'bool', 'Marker (>)': 'bool', 'Marker (*)': 'bool', 'Marker (@)': 'bool'})

    # Change column type to float64 for columns: 'Energie (keV)', 'Intensité (%)' and 4 other columns
    df = df.astype({'Energie (keV)': 'float64', 'Intensité (%)': 'float64', 'LD Energie (mBq/g   )': 'float64', 'LD nucléide (mBq/g   )': 'float64', 'Activité (mBq/g   )': 'float64', 'SD Energie (mBq/g   )': 'float64'})

    # Replace gaps forward from the previous valid value in: 'Nom du nucléide'
    df = df.fillna({'Nom du nucléide': df['Nom du nucléide'].ffill()})
    
    return df

In [ ]:
content_s5 = sections_content[sections_titles[4]]
header_s5 = extract_header_s5(content_s5)
data_s5 = extract_data_s5(content_s5, header_s5)

data_s5

### S6 - RAPPORT LIMITES DE DETECTION  ISO 11929

In [ ]:
def extract_header_s6(content):
    header = []
    matches = re.findall(s6_header_pattern, content)

    l1 = re.findall(s6_word_column_pattern, matches[0][0])
    l2 = re.findall(s6_word_column_pattern, matches[0][1])
    
    # Fusion en partant de la fin
    for a, b in zip(reversed(l1), reversed(l2)):
        header.insert(0, f"{a} {b}".strip())

    # Si l1 est plus longue, on conserve le début restant
    if len(l1) > len(l2):
        header = l1[: len(l1) - len(l2)] + header

    return header

In [ ]:
def extract_data_s6(content, header):
    matches = re.findall(s6_nucleide_line_pattern, content)
    df = pd.DataFrame(matches, columns=header)
    
    # Change column type to category for column: 'Nucléide'
    df = df.astype({'Nucléide': 'category'})
    
    # Change column type to float64 for columns: 'LD', 'SD' and 6 other columns
    df = df.astype({'LD': 'float64', 'SD': 'float64', 'Limite Basse': 'float64', 'Limite Haute': 'float64', 'Moyenne Activité': 'float64', 'pondérée Incert.': 'float64', 'Meilleure Activité': 'float64', 'Estimation Incert.': 'float64'})

    return df

In [ ]:
content_s6 = sections_content[sections_titles[5]]
header_s6 = extract_header_s6(content_s6)
data_s6 = extract_data_s6(content_s6, header_s6)

####### DATA ##########
data_s6

# Zone de Test